# Image Emotion Detection using ResNet50

## Imports

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import (
accuracy_score,
confusion_matrix,
classification_report,
roc_curve,
auc
)

## Paths & Device

In [ ]:
ROOT_DIR = r"C:\Users\USER\PycharmProjects\DSGP15_Project\ml-models\dataset\Dataset"


train_dir = os.path.join(ROOT_DIR, "train")
val_dir = os.path.join(ROOT_DIR, "val")
test_dir = os.path.join(ROOT_DIR, "test")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Image Transformations

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32


train_transform = transforms.Compose([
transforms.Resize((IMG_SIZE, IMG_SIZE)),
transforms.RandomHorizontalFlip(),
transforms.ToTensor(),
transforms.Normalize(
mean=[0.485, 0.456, 0.406],
std=[0.229, 0.224, 0.225]
)
])


val_test_transform = transforms.Compose([
transforms.Resize((IMG_SIZE, IMG_SIZE)),
transforms.ToTensor(),
transforms.Normalize(
mean=[0.485, 0.456, 0.406],
std=[0.229, 0.224, 0.225]
)
])

## Dataset & DataLoader

In [ ]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_test_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transform)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


print("Classes:", train_dataset.classes) # ['happy', 'sad']

## Load ResNet50

In [ ]:
model = models.resnet50(pretrained=True)


# Freeze convolutional layers
for param in model.parameters():
param.requires_grad = False


# Replace final layer for binary classification
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)